In [1]:
import multiprocessing
multiprocessing.set_start_method("spawn", force=True)

import polars as pl
import numpy as np
import matplotlib.pyplot as plt

In [1]:
import torch
# check if GPU is available
if torch.cuda.is_available():
    device = torch.device("cuda")
else:
    device = torch.device("cpu")
print(f"Using device: {device}")

Using device: cpu


In [ ]:
datapath = '../data/2010-2011 Solar home electricity data.csv'
# skip the first line in csv and read the next line as column
# then read the rest of the file and store as dataframe
df = pl.read_csv(datapath, skip_rows=1)
print(df)
print(df.columns)
episode_num = "episode_01"

In [2]:
datapath = '../data/2012-2013 Solar home electricity data v2.csv'
# skip the first line in csv and read the next line as column
# then read the rest of the file and store as dataframe
df = pl.read_csv(datapath, skip_rows=1)
print(df)
print(df.columns)
episode_num = "episode_02"

shape: (268_557, 54)
┌──────────┬───────────┬──────────┬──────────────────────┬───┬───────┬───────┬───────┬─────────────┐
│ Customer ┆ Generator ┆ Postcode ┆ Consumption Category ┆ … ┆ 23:00 ┆ 23:30 ┆ 0:00  ┆ Row Quality │
│ ---      ┆ Capacity  ┆ ---      ┆ ---                  ┆   ┆ ---   ┆ ---   ┆ ---   ┆ ---         │
│ i64      ┆ ---       ┆ i64      ┆ str                  ┆   ┆ f64   ┆ f64   ┆ f64   ┆ str         │
│          ┆ f64       ┆          ┆                      ┆   ┆       ┆       ┆       ┆             │
╞══════════╪═══════════╪══════════╪══════════════════════╪═══╪═══════╪═══════╪═══════╪═════════════╡
│ 1        ┆ 3.78      ┆ 2076     ┆ CL                   ┆ … ┆ 0.0   ┆ 0.0   ┆ 1.081 ┆ null        │
│ 1        ┆ 3.78      ┆ 2076     ┆ GC                   ┆ … ┆ 0.651 ┆ 0.09  ┆ 0.068 ┆ null        │
│ 1        ┆ 3.78      ┆ 2076     ┆ GG                   ┆ … ┆ 0.0   ┆ 0.0   ┆ 0.0   ┆ null        │
│ 1        ┆ 3.78      ┆ 2076     ┆ CL                   ┆ … ┆ 0.0   ┆

In [3]:
# we can get the training and testing customers from the csv file
training_customers = np.loadtxt('../data/training_customers.csv', dtype=int)
testing_customers = np.loadtxt('../data/testing_customers.csv', dtype=int)

In [ ]:
# alternatively, get all the unique customers as their own dataframes
customers = df['Customer'].unique()
# pick 80% of the random customers as training data
training_customers = np.random.choice(customers, int(0.8*len(customers)), replace=False)
# the rest of the customers are testing data
testing_customers = np.setdiff1d(customers, training_customers)

In [ ]:
# save the customers number to a csv file
np.savetxt('../data/training_customers.csv', training_customers, fmt='%s')
np.savetxt('../data/testing_customers.csv', testing_customers, fmt='%s')

In [5]:

from helper import transform_polars_df
# loop through each customer and use transform_polars_df to get the dataframe and store it in a list call dataset
training_dataset = []
for customer in training_customers:
    customer_df = df.filter(pl.col('Customer') == customer)
    try:
        newcustomerdf = transform_polars_df(customer_df, import_energy_price=0.23, export_energy_price=0.015, price_periods="7am – 10am | 4pm – 9pm", default_import_energy_price=0.15, default_export_energy_price=0.01)
    except Exception as e:
        print(f"Error with customer as training dataset: {customer}")
        print(e)
        break
    training_dataset.append(newcustomerdf)

testing_dataset = []
for customer in testing_customers:
    customer_df = df.filter(pl.col('Customer') == customer)
    try:
        newcustomerdf = transform_polars_df(customer_df, import_energy_price=0.23, export_energy_price=0.015, price_periods="7am – 10am | 4pm – 9pm", default_import_energy_price=0.15, default_export_energy_price=0.01)
    except Exception as e:
        print(f"Error with customer as testing dataset: {customer}")
        print(e)
        break
    testing_dataset.append(newcustomerdf)

In [ ]:
# provide std and mean on the different columns of the training dataset
testdf = testing_dataset[25]
# drop timestamp and time columns
testdf = testdf.drop(['Timestamp', 'Time'])
print(testdf.describe())

In [6]:
import gymnasium as gym

from stable_baselines3.common.vec_env import DummyVecEnv, SubprocVecEnv
from stable_baselines3.common.env_checker import check_env
from EnergySimEnv import SolarBatteryEnv
from helper import make_env

testing_env_fns = [make_env(ds) for ds in testing_dataset]

num_step = None # pick the number of step for the simulation
test_envs = [env_fn(num_step) for env_fn in testing_env_fns]

/usr/local/lib/python3.10/dist-packages/gymnasium/spaces/box.py:236: UserWarning: WARN: Box low's precision lowered by casting to float32, current low.dtype=float64
  gym.logger.warn(
/usr/local/lib/python3.10/dist-packages/gymnasium/spaces/box.py:306: UserWarning: WARN: Box high's precision lowered by casting to float32, current high.dtype=float64
  gym.logger.warn(


In [ ]:
# Create a list of environment creation functions to build a vectorized environment.
training_env_fns = [make_env(ds) for ds in training_dataset]
#training_vec_env = DummyVecEnv(training_env_fns)

num_total_steps = len(training_dataset[0])
print(f"Total number of steps possible in training dataset: {num_total_steps}")

train_envs = [env_fn(num_step) for env_fn in training_env_fns]

In [ ]:
# combine the test_envs and train_envs into a single list
combined_envs = test_envs + train_envs

In [7]:
selected_list = test_envs
if selected_list is test_envs:
    env_type = "test"
elif selected_list is train_envs:
    env_type = "train"
else:
    env_type = "combined"

In [ ]:
from decision import Agent, run_episodes_parallel
rule_agent_kwargs = {
    'algorithm': 'rule'
}

# run episodes in the list in parallel using the rule-based agent on the training environments
episode_logs = run_episodes_parallel(Agent, selected_list, agent_kwargs=rule_agent_kwargs, max_workers=12, use_notebook_tqdm=False)

In [ ]:
dfs_with_id = [df.with_columns(pl.lit(i).alias("episode_id")) for i, df in enumerate(episode_logs)]
rule_all_logs = pl.concat(dfs_with_id)
file_name = f"../data/rule_{env_type}_{episode_num}_logs.parquet"
rule_all_logs.write_parquet(file_name)

In [ ]:
from decision import Agent, run_episodes_parallel, run_single
# Initialize environments and SDP agent parameters
sdp_agent_kwargs = {
    'algorithm': 'sdp',
    'soc_resolution': 20,
    'action_resolution': 41,  # best to be 2*soc_resolution + 1
    'degradation_model': 'linear', # the other option being static degradation 'static'
    'linear_deg_cost_p_kwh': 0.2 # only needed if using linear
}

# Run a single episode for timing test
#sdp_single_log = run_single(Agent, combined_envs[0], agent_kwargs=sdp_agent_kwargs, render=False, display_progress=True)


# Run all episodes in parallel
sdp_episode_logs = run_episodes_parallel(Agent, selected_list, agent_kwargs=sdp_agent_kwargs, max_workers=12, use_notebook_tqdm=False)

In [ ]:
dfs_with_id = [df.with_columns(pl.lit(i).alias("episode_id")) for i, df in enumerate(sdp_episode_logs)]
sdp_all_logs = pl.concat(dfs_with_id)
file_name = f"../data/sdp_{env_type}_{episode_num}_logs.parquet"
sdp_all_logs.write_parquet(file_name)

In [ ]:
from decision import Agent, run_episodes_parallel, run_single
mrdp_agent_kwargs = {
    'algorithm': 'mrdp',
    'soc_resolution': 20,           # fallback/default for single-horizon
    'action_resolution': 41,        # fallback/default for single-horizon
    'degradation_model': 'linear',  # or 'static'
    'linear_deg_cost_p_kwh': 0.2,   # only needed if using linear
    'subhorizon_specs': [
        {
            'start': 0,
            'length': 12,           # e.g. 6 hours at 30-min steps
            'soc_res': 20,          # fine SoC discretization
            'action_res': 11,       # fine action discretization
            'step_duration': 0.5    # hours per step (30 min)
        },
        {
            'start': 12,
            'length': 36,           # e.g. 18 hours at 30-min steps
            'soc_res': 8,           # coarse SoC discretization
            'action_res': 5,        # coarse action discretization
            'step_duration': 1.0    # hours per step (1 hour)
        }
    ]
}

#sdp_single_log = run_single(Agent, combined_envs[0], agent_kwargs=mrdp_agent_kwargs, render=False, display_progress=True)

# Run all episodes in parallel using MRDP

mrdp_episode_logs = run_episodes_parallel(
    Agent, selected_list, agent_kwargs=mrdp_agent_kwargs, max_workers=12, use_notebook_tqdm=False
)

In [ ]:
dfs_with_id = [df.with_columns(pl.lit(i).alias("episode_id")) for i, df in enumerate(mrdp_episode_logs)]
mrdp_episode_logs = pl.concat(dfs_with_id)
file_name = f"../data/mrdp_{env_type}_{episode_num}_logs.parquet"
mrdp_episode_logs.write_parquet(file_name)

In [ ]:
from decision import Agent, run_sb3_model_on_vec_env
from stable_baselines3 import PPO, A2C, DDPG, SAC, TD3

# (only needed if you ever switch to SubprocVecEnv on Linux/notebooks)
multiprocessing.set_start_method("forkserver", force=True)

from stable_baselines3.common.vec_env import SubprocVecEnv
if env_type is "test":
    vec_env = SubprocVecEnv(testing_env_fns)
elif env_type is "train":
    vec_env = SubprocVecEnv(training_env_fns)


from helper import flatten_episode_data

In [ ]:
sac_model = SAC.load("../models/sac_model.zip")
# Run test episodes in parallel
sac_episode_logs = run_sb3_model_on_vec_env(sac_model, vec_env)

sac_logs = flatten_episode_data(sac_episode_logs)
file_name = f"../data/sac_{env_type}_{episode_num}_logs.parquet"
sac_logs.write_parquet(file_name)

In [ ]:
PPO_model = PPO.load("../models/ppo_model.zip")
ppo_episode_logs = run_sb3_model_on_vec_env(PPO_model, vec_env)

ppo_logs = flatten_episode_data(ppo_episode_logs)
file_name = f"../data/ppo_{env_type}_{episode_num}_logs.parquet"
ppo_logs.write_parquet(file_name)

In [ ]:
a2c_model = A2C.load("../models/a2c_model.zip")
a2c_episode_logs = run_sb3_model_on_vec_env(a2c_model, vec_env)

a2c_logs = flatten_episode_data(a2c_episode_logs)
file_name = f"../data/a2c_{env_type}_{episode_num}_logs.parquet"
a2c_logs.write_parquet(file_name)

In [ ]:
ddpg_model = DDPG.load("../models/ddpg_model.zip")
ddpg_episode_logs = run_sb3_model_on_vec_env(ddpg_model, vec_env)
ddpg_logs = flatten_episode_data(ddpg_episode_logs)
file_name = f"../data/ddpg_{env_type}_{episode_num}_logs.parquet"
ddpg_logs.write_parquet(file_name)

In [ ]:
td3_model = TD3.load("../models/td3_model.zip")
td3_episode_logs = run_sb3_model_on_vec_env(td3_model, vec_env)
td3_logs = flatten_episode_data(td3_episode_logs)
file_name = f"../data/td3_{env_type}_{episode_num}_logs.parquet"
td3_logs.write_parquet(file_name)

In [8]:
from decision import Agent, run_episodes_parallel, run_single
from decision_transformer import DecisionTransformer
import json

import torch
# check if GPU is available
if torch.cuda.is_available():
    device = torch.device("cuda")
else:
    device = torch.device("cpu")
print(f"Using device: {device}")

with open('../models/decision_transformer_model_kwargs.json', 'r') as f:
    model_kwargs = json.load(f)

model = DecisionTransformer(**model_kwargs)

model.load_state_dict(torch.load('../models/dt_model.pt', map_location=device))
model.return_scale = 1.0  # or whatever was used during training
model.eval()
rtg = 100.0

dt_agent_kwargs = {
    'algorithm': 'dt',
    'model': model.to(device),
    'rtg_value': rtg
}

# check for nan in model parameters
import math
bad_params = [name for name, p in model.named_parameters() if torch.isnan(p).any() or torch.isinf(p).any()]
print("bad params:", bad_params)

Using device: cpu
bad params: []


In [ ]:
episode_log = run_episodes_parallel(Agent, test_envs, agent_kwargs=dt_agent_kwargs, max_workers=2, use_notebook_tqdm=False)

dfs_with_id = [df.with_columns(pl.lit(i).alias("episode_id")) for i, df in enumerate(episode_log)]
dt_logs = pl.concat(dfs_with_id)
file_name = f"../data/dt_rtg{int(rtg)}_{env_type}_{episode_num}_logs.parquet"
dt_logs.write_parquet(file_name)